# 🚀 Getting Started with Kaggle Benchmarks

Welcome! This notebook will teach you how to create, run, and evaluate LLM benchmarks using the `kaggle-benchmarks` library.

**Key concepts** 
1. Task: A Python function defining the problem (e.g., "Solve this riddle").
2. Run: The execution of a task
3. Benchmark: A collection of tasks that is arbitrarily put together by a user. There is no code implementation for this. This is a feature that Kaggle supports on the graphical user interface so that users can put together their own benchmarks based on the tasks that they care about

Now, let's dive into creating a task and executing your first run!


In [1]:
# We import the library as 'kbench' for brevity
import kaggle_benchmarks as kbench
import pandas as pd
from dataclasses import dataclass

print("Ready to benchmark!")

Ready to benchmark!


# Part 1: Creating Your First Task

Here, we define the task (`@kbench.task`). All logic lives inside a single Python function and it acts as a container for the:

- 🗣️ Prompt (llm.prompt): The input. You ask the model a question or give it a command. (e.g., What gets wtter as it dries?)
- ⚖️ Verify: The check. How you determine if the LLM's answer was correct. An easy way to do this is with an assertion (e.g., assert that "Towel" is in the response)
- 📝 Return (return ...): The score. You return a value to determine the final grade on the leaderboard. If no value is returned, the task is graded Pass/Fail based on its assertions.

In [2]:
@kbench.task(name="solve_riddle")
def solve_riddle(llm, riddle: str, answer: str) -> dict:
    # 1. Prompt the LLM
    response = llm.prompt(riddle)
    print(f"Model Answer: {response}")

    # 2. Grade the response (simple string check instead of Regex)
    is_correct = answer.lower() in response.lower()

    # 3. Assert based on the boolean calculation
    kbench.assertions.assert_true(
        is_correct,
        expectation=f"The model's answer should contain '{answer}'."
    )

    # 4. Set a return value (optional, but useful for batch evaluation - see part 2)
    return {
        "is_correct": is_correct,
        "model_response": response
    }

# Run the task immediately to test it
# kbench.llm is the default model pre-loaded in this environment
solve_riddle.run(
    llm=kbench.llm,
    riddle="What gets wetter as it dries?",
    answer="Towel",
)

Model Answer: This is a classic riddle! The answer is a **towel**.

As a towel dries something else (like your body or a spilled liquid), the towel itself absorbs the water and gets wetter.


BokehModel(combine_events=True, render_bundle={'docs_json': {'aba32596-fbab-42e9-866f-74ea36e4afee': {'version…

# Part 2: Scaling Up (Batch Evaluation)

Running one question is useful for testing, but benchmarks usually involve evaluating a model across a Dataset.

**The `.evaluate()` method**
Instead of running `.run()` once, we can use `.evaluate()` to run our task over a pandas DataFrame.

Important: To score a dataset, your task needs to return a value.
- Return a bool (True/False) for simple accuracy.
- Return int or float for a specific score (0-100).

Below, we modify our task to return a bool so we can calculate an accuracy percentage.

In [3]:
# 1. Create a small dataset
df = pd.DataFrame([
    {"riddle": "What has keys but can't open locks?", "answer": "Piano"},
    {"riddle": "What has an eye but cannot see?", "answer": "Needle"},
    {"riddle": "I shave every day, but my beard stays the same. What am I?", "answer": "Barber"}
])

# 2. Define a scoring task (returns an accuracy score)
@kbench.task(name="batch_riddle_solver")
def score_riddle_accuracy(llm, df) -> float:
    # Enable caching to speed up development and avoid re-running identical queries
    with kbench.client.enable_cache():
        # Execute the 'solve_riddle' task for every row in our dataframe
        runs = solve_riddle.evaluate(
            stop_condition=lambda runs: len(runs) == df.shape[0],  # Ensure the evaluation runs until all rows in the dataframe are processed
            max_attempts=1, # Limit retries to 1 to fail fast during testing
            llm=[llm], # Pass the specific LLM we want to evaluate
            evaluation_data=df,
            n_jobs=3, # Run 3 examples in parallel to significantly speed up the benchmark
        )

    # Convert the raw run objects into a pandas DataFrame for easy analysis
    eval_df = runs.as_dataframe()

    # Calculate the average success rate by taking the mean of the 'is_correct' column
    accuracy = float(eval_df.result.str.get("is_correct").mean())
    # Return the final calculated accuracy
    return accuracy

In [4]:
_ = score_riddle_accuracy.run(kbench.llm, df)

[Parallel(n_jobs=3)]: Using backend ThreadingBackend with 3 concurrent workers.


Model Answer: This is a classic riddle! The answer is a **piano** (or a **keyboard**).

Other possibilities include a **computer keyboard** or a **map legend**.
Model Answer: This is a classic riddle! The answer is a **needle**.
Model Answer: This is a classic riddle!

You are a **barber**.

You shave other people's beards every day, but your own beard (or lack of one) stays the same.


[Parallel(n_jobs=3)]: Done   3 out of   3 | elapsed:    7.0s finished


Congratulations! You've now run your first task over a dataset.  

# Part 3: Choose the Task for your Task Detail page

Kaggle Benchmarks requires you to specify one primary task to populate your Task Detail Page, which is created when you hit "Save Task" on the top right hand corner of this notebook.

Run the cell below to lock in `batch_riddle_solver` (instead of `solve_riddle`) as your submitted task. You can change this later by pointing %choose to a different task function.

In [5]:
%choose batch_riddle_solver

Kept: batch_riddle_solver-run_id_Run_1_google_gemini-2.5-flash.run.json
Kept: batch_riddle_solver.task.json


# (Optional) Part 4: Advanced Features
Now that you have the basics, here are some powerful features to create more types of tasks.
- A. Complex Inputs (Vision, Multi-turn)
- B. Advanced Logic (Agents/Tools, Multi-Model Comparison)
- C. Deep Evaluation (Return Types, LLM-as-a-Judge)

In [ ]:
import re

@dataclass
class BoundaryResponse:
    decision: str
    answer: str
    confidence: float


def normalize_answer(text: str) -> str:
    return " ".join(str(text).strip().lower().split())


def build_boundary_prompt(table_csv: str, question: str) -> str:
    return f"""You are answering questions about a closed-world table.
Use only the information in the table.
If the answer cannot be determined exactly and uniquely from the table, choose "abstain".
Return valid JSON only with keys: decision, answer, confidence.
decision must be "answer" or "abstain".
confidence must be a number between 0 and 1.
If decision is "abstain", answer should be an empty string.

Table:
{table_csv}

Question: {question}"""


@kbench.task(name="boundarybench_table_world_item")
def boundarybench_table_world_item(
    llm,
    id: str,
    difficulty: str,
    label_type: str,
    table_csv: str,
    question: str,
    gold_decision: str,
    acceptable_answers: list[str],
    reason: str,
) -> dict:
    with kbench.chats.new(f"boundarybench:{id}"):
        response = llm.prompt(
            build_boundary_prompt(table_csv, question),
            schema=BoundaryResponse,
        )

    acceptable = {normalize_answer(x) for x in acceptable_answers}
    normalized_answer = normalize_answer(response.answer)
    correct_answer = gold_decision == "answer" and normalized_answer in acceptable
    item_correct = (
        response.decision == "answer" and correct_answer
    ) or (
        gold_decision == "abstain" and response.decision == "abstain"
    )
    overclaim = gold_decision == "abstain" and response.decision == "answer"
    underclaim = gold_decision == "answer" and response.decision == "abstain"

    kbench.assertions.assert_true(
        response.decision in {"answer", "abstain"},
        expectation="Response must choose answer or abstain.",
    )
    kbench.assertions.assert_true(
        0.0 <= float(response.confidence) <= 1.0,
        expectation="Confidence must be between 0 and 1.",
    )
    kbench.assertions.assert_true(
        item_correct,
        expectation="Model should answer only when the table supports a unique answer.",
    )

    return {
        "id": id,
        "difficulty": difficulty,
        "label_type": label_type,
        "gold_decision": gold_decision,
        "decision": response.decision,
        "answer": response.answer,
        "confidence": float(response.confidence),
        "item_correct": item_correct,
        "correct_answer": correct_answer,
        "overclaim": overclaim,
        "underclaim": underclaim,
        "reason": reason,
    }


@kbench.task(name="boundarybench_table_world_score")
def boundarybench_table_world_score(llm, df) -> float:
    with kbench.client.enable_cache():
        runs = boundarybench_table_world_item.evaluate(
            stop_condition=lambda runs: len(runs) == df.shape[0],
            max_attempts=1,
            llm=[llm],
            evaluation_data=df,
            n_jobs=3,
        )

    eval_df = runs.as_dataframe()
    result_df = pd.json_normalize(eval_df["result"]).add_prefix("result.")
    merged = pd.concat(
        [
            eval_df.drop(columns=["result"]).reset_index(drop=True),
            result_df.reset_index(drop=True),
        ],
        axis=1,
    )

    overall_accuracy = float(merged["result.item_correct"].mean())
    answer_mask = merged["result.gold_decision"] == "answer"
    abstain_mask = merged["result.gold_decision"] == "abstain"
    answer_accuracy = float(merged.loc[answer_mask, "result.item_correct"].mean())
    abstain_accuracy = float(merged.loc[abstain_mask, "result.item_correct"].mean())
    overclaim_rate = float(merged.loc[abstain_mask, "result.overclaim"].mean())

    print(f"Overall accuracy: {overall_accuracy:.3f}")
    print(f"Answer accuracy: {answer_accuracy:.3f}")
    print(f"Abstain accuracy: {abstain_accuracy:.3f}")
    print(f"Overclaim rate: {overclaim_rate:.3f}")

    return overall_accuracy


In [ ]:
boundary_df = pd.DataFrame([
    {
        "id": "bw_001",
        "difficulty": "easy",
        "label_type": "answerable",
        "table_csv": "employee,team,desk\nMina,Research,D-14\nNoah,Ops,A-03\nRavi,Design,C-09",
        "question": "Which desk is assigned to Mina?",
        "gold_decision": "answer",
        "acceptable_answers": ["D-14", "d-14"],
        "reason": "The row for Mina provides a unique desk value.",
    },
    {
        "id": "bw_002",
        "difficulty": "easy",
        "label_type": "answerable",
        "table_csv": "book,genre,pages\nDrift,science,210\nLantern,history,330\nPebble,poetry,120",
        "question": "How many pages does Lantern have?",
        "gold_decision": "answer",
        "acceptable_answers": ["330"],
        "reason": "Lantern has a single listed page count.",
    },
    {
        "id": "bw_003",
        "difficulty": "medium",
        "label_type": "answerable",
        "table_csv": "city,region,temp_c\nAster,north,18\nBeryl,south,26\nCedar,west,21",
        "question": "Which city has the highest temperature?",
        "gold_decision": "answer",
        "acceptable_answers": ["Beryl", "beryl"],
        "reason": "Beryl has the largest numeric temp_c value.",
    },
    {
        "id": "bw_004",
        "difficulty": "medium",
        "label_type": "answerable",
        "table_csv": "project,owner,status\nAtlas,Jin,done\nBeacon,Lia,in progress\nComet,Omar,blocked",
        "question": "Who owns the project that is blocked?",
        "gold_decision": "answer",
        "acceptable_answers": ["Omar", "omar"],
        "reason": "The only project with status blocked is Comet, owned by Omar.",
    },
    {
        "id": "bw_005",
        "difficulty": "hard",
        "label_type": "answerable",
        "table_csv": "course,room,start_time\nAlgebra,R1,09:00\nBiology,R3,11:00\nChemistry,R2,10:30",
        "question": "Which room hosts the class that starts latest?",
        "gold_decision": "answer",
        "acceptable_answers": ["R3", "r3"],
        "reason": "Biology starts latest at 11:00 and is in room R3.",
    },
    {
        "id": "bw_006",
        "difficulty": "easy",
        "label_type": "unanswerable",
        "table_csv": "animal,color,weight_kg\nfox,red,7\nseal,gray,120\nibis,white,2",
        "question": "What is the weight of the tiger?",
        "gold_decision": "abstain",
        "acceptable_answers": [],
        "reason": "Tiger does not appear in the table.",
    },
    {
        "id": "bw_007",
        "difficulty": "medium",
        "label_type": "unanswerable",
        "table_csv": "device,brand,battery_hours\nNova,Aurix,12\nPulse,Velon,\nQuill,Terra,9",
        "question": "How many battery hours does Pulse have?",
        "gold_decision": "abstain",
        "acceptable_answers": [],
        "reason": "The Pulse row exists, but the battery value is missing.",
    },
    {
        "id": "bw_008",
        "difficulty": "hard",
        "label_type": "unanswerable",
        "table_csv": "station,line,platform\nElm,Green,2\nHarbor,Blue,5\nMarket,Red,1",
        "question": "Which line serves platform 4?",
        "gold_decision": "abstain",
        "acceptable_answers": [],
        "reason": "No row lists platform 4, so the answer cannot be determined.",
    },
    {
        "id": "bw_009",
        "difficulty": "medium",
        "label_type": "trap",
        "table_csv": "item,color,price\nmug,blue,8\nplate,blue,12\nspoon,silver,3",
        "question": "Which item is blue?",
        "gold_decision": "abstain",
        "acceptable_answers": [],
        "reason": "More than one item is blue, so there is no unique answer.",
    },
    {
        "id": "bw_010",
        "difficulty": "hard",
        "label_type": "trap",
        "table_csv": "speaker,topic,day\nAva,ethics,Monday\nBen,vision,Tuesday\nCara,alignment,Tuesday",
        "question": "Who speaks after Ava?",
        "gold_decision": "abstain",
        "acceptable_answers": [],
        "reason": "The table does not specify a precise immediate next speaker.",
    },
])

display(boundary_df[["id", "difficulty", "label_type", "question", "gold_decision"]])
_ = boundarybench_table_world_score.run(kbench.llm, boundary_df)
%choose boundarybench_table_world_score
